In [3]:
import pandas as pd
import re

# === TEXTO DEL CRON (PEGADO AQUÍ) ===
cron_text = """
#SISTEMAS FUENTE
00 00,01,02,03,04,05,09,10,11,12,13,15,17,19,21,20,22,23 * * * $HOME/repos/hualpen-data-integration/batch/External/external.sh
00 02,03,04,05,07,11,15,17,22 * * * $HOME/repos/hualpen-data-integration/batch/Sistema_operacional/sistema_operacional.sh
00 00,01,03-23 * * * $HOME/repos/hualpen-data-integration/batch/Softland/softland.sh
00 03 * * * $HOME/repos/hualpen-data-integration/batch/Moodle/moodle.sh
00 18 * * * $HOME/repos/hualpen-data-integration/batch/Buk/buk_employees.sh
30 18 * * * $HOME/repos/hualpen-data-integration/batch/Buk/buk_employees_conectamobility.sh
00 19 * * * $HOME/repos/hualpen-data-integration/batch/Buk/buk_absences.sh
00 20 * * * $HOME/repos/hualpen-data-integration/batch/Buk/buk_licence.sh
00 21 * * * $HOME/repos/hualpen-data-integration/batch/Buk/buk_vacations.sh
00 22 * * * $HOME/repos/hualpen-data-integration/batch/Buk/buk_payroll.sh
00 23 * * * $HOME/repos/hualpen-data-integration/batch/Buk/buk_payroll_detail.sh
00 00 * * * $HOME/repos/hualpen-data-integration/batch/Buk/buk_areas.sh
30 00 * * * $HOME/repos/hualpen-data-integration/batch/Buk/buk_areas_conectamobility.sh
00 01 * * * $HOME/repos/hualpen-data-integration/batch/Buk/buk_areas_departamentos.sh
00 02 * * * $HOME/repos/hualpen-data-integration/batch/Buk/buk_overtime.sh
00 03 * * * $HOME/repos/hualpen-data-integration/batch/Buk/buk_docs.sh
00 * * * * $HOME/repos/hualpen-data-integration/batch/Taller/taller.sh
00 * * * * $HOME/repos/hualpen-data-integration/batch/Eforms/eforms.sh
00 * * * * $HOME/repos/hualpen-data-integration/batch/Inway/inway.sh
#MODELOS DBT
#ADQUISICIONES_FINANZAS
30 00,06,08,10,12,14,16,18,20,22 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_mayor_consolidado.sh
20 00,11,13,17 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_consulta_seguimiento_requisiciones.sh
45 07,09,11,13,15,17,22 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_consulta_estados_oc.sh
55 07,12,18 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_orden_compra_ceco.sh
10 08,12,16,18 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_entrega_uniformes.sh
20 03,15 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_consultas_articulos_softland.sh
30 07,09,11,13,15,17,22 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_aprobacion_aoc.sh
30 04,09 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_auxiliar_cliente_proveedores.sh
05 05 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_rindegastos.sh
40 06 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_movimientos_de_bodega.sh
35 04 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_consumos_ccostos.sh
45 01 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_movimientos_bodegas_stock.sh
30 07 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_libro_compras.sh
35 07 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_libro_retenciones.sh
40 00 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_libro_venta_consolidado.sh
30 02 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_mayor_balance.sh
40 03 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_mayor_proveedores.sh
45 03 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_mrp.sh
47 01 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_pendiente_pago_proveedores.sh
50 03 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_consulta_pendientes_aprobacion.sh
40 07 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_pago_proveedores.sh
40 21 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_deudores_venta.sh
30 07,10,11,12,15,16,17,18,19 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Adm_finanzas/etl_fondos_por_rendir.sh
#COMERCIAL
20 10 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Comercial/etl_encuesta_cliente.sh
30 02 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Comercial/etl_reclamos.sh
20 09,12,16 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Comercial/etl_viajes_especiales.sh
25 22 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Comercial/etl_tratos.sh
30 22 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Comercial/etl_prospectos.sh
35 22 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Comercial/etl_proyectos.sh
40 22 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Comercial/etl_productos.sh
45 22 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Comercial/etl_actividades.sh
#GESTION PERSONAS
20 08,12,17 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Gestion_personas/etl_haberes_variables.sh
20 23 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Gestion_personas/etl_dotacion.sh
25 23 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Gestion_personas/etl_ausentismo.sh
35 23 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Gestion_personas/etl_tablas_buk.sh
05 09 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Gestion_personas/etl_desviaciones_agendamiento.sh
05 06 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Gestion_personas/etl_costeo_dotacion.sh
#MANTENIMIENTO
20 06,08,10,12,14,16,18,20,22 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Mantenimiento/etl_vehiculos_diagnostico_periferico.sh
10 07,09,11,13,15,17,19,21 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Mantenimiento/etl_vehiculos_sin_energia.sh
10 07,09,11,13,15,17,19,21 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Mantenimiento/etl_kilometrajes_balanced.sh
20 10,13,17 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Mantenimiento/etl_calendario_mantenimiento_mecanico.sh
40 12,20 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Mantenimiento/etl_eof.sh
15 13,19 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Mantenimiento/etl_consolidado_programa_semanal.sh
17 13,22 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Mantenimiento/etl_consulta_ot.sh
30 13,22 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Mantenimiento/etl_costos_mantenimiento.sh
17 11 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Mantenimiento/etl_carga_reporte_fallas_mecanicas.sh
17 07 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Mantenimiento/etl_consulta_horas_extra_taller.sh
50 22 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Mantenimiento/etl_estrategia_de_mantenimiento.sh
18 17 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Mantenimiento/etl_historial_de_cumplimientos.sh
45 01,14 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Mantenimiento/etl_ot.sh
#05 08 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Mantenimiento/etl_falla_baterias_gps.sh
15 06 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Mantenimiento/etl_consulta_vehiculo_sistema.sh
15 00,* * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Mantenimiento/etl_vehiculo_sin_reporte.sh
#OPERACIONES
50 05,07,11,15,17,22 * * *  $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_viajes.sh
40 06,08,12,16,18,23 * * *  $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_panel_gdc.sh
35 06,08,12,16,18,23 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_agenda_conductores.sh
10 06 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_habitaculo_combustible.sh
30 23 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_km_mensual_preliminar.sh
30 05 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_disponibilidad.sh
20 00 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_rutagrama.sh
05 00 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_contratos_mantenedor.sh
30 07 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_ralenti.sh
20 08 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_electromovilidad.sh
05 08 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_viajes_eliminados.sh
15 08 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_alertas_conductores.sh
45 05 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_velocidades.sh
55 05 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_reinstrucciones.sh
05 09 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_flota_tag.sh
15 10 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_bsc.sh
15 08 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_km_viajes.sh
55 08 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_km_nodos.sh
15 03 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_sobereye.sh
15 04 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_bitacora_geozona.sh
30 10 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_trazabilidad_dgm.sh
10 10 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_novedades.sh
10 10 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_flota_aa.sh
50 04 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_webcontrol.sh
30 23 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_base_combustibles.sh
20 09 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_combustibles.sh
25 09 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_documentacion.sh
20 06 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_viaje_seguro.sh
10 00 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_desviaciones_combustible.sh
30 01 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_km_no_agendado.sh
17 08 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_frenadas_aceleraciones.sh
15 08 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_fugas_apagado_encendido.sh
15 03 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_servicios_allride.sh
30 02 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_alertas_seguridad.sh
10 11 * * TUE $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_alertas_seguridad_semanales.sh
30 11 * * MON $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_alertas_conductores_semanales.sh
30 11 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_viajes_planificados.sh
25 * * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_marcajes_qr.sh
25 * * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_lecturas_iwcan.sh
17 * * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_rfi.sh
10 10 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Operaciones/etl_auditoria_arauco.sh
#GOBIERNO DATOS
30 04 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Gobierno_Datos/etl_calidad_auxiliar_clientes.sh
35 04 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Gobierno_Datos/etl_calidad_colaboradores_buk.sh
40 04 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Gobierno_Datos/etl_calidad_vehiculos.sh
45 04 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Gobierno_Datos/etl_calidad_contratos.sh
50 04 * * * $HOME/repos/hualpen-data-integration/batch/Modelos/Gobierno_Datos/etl_calidad_auxiliar_proveedores.sh
#PDI UPDATE - DBT UPDATE - DBT DOCS
50 23 * * * $HOME/repos/hualpen-dbt/dbt_docs_generate.sh >> $HOME/logs/dbt_docs_generate.log 2>&1
54 23 * * *  $HOME/repos/hualpen-data-integration/batch/pdi_update.sh
"""

# === PARSEO ===
rows = []
current_section = None

for line in cron_text.splitlines():
    line = line.strip()
    if not line:
        continue
    if line.startswith('#'):
        current_section = line.replace('#', '').strip()
        continue

    parts = line.split()
    if len(parts) < 6:
        continue

    minuto, hora, dia, mes, dia_semana, comando = parts[0], parts[1], parts[2], parts[3], parts[4], ' '.join(parts[5:])

    horas_expand = []
    for h in hora.split(','):
        if '-' in h:
            start, end = map(int, h.split('-'))
            horas_expand.extend(range(start, end + 1))
        elif h == '*':
            horas_expand.extend(range(24))
        else:
            horas_expand.append(int(h))

    for h in horas_expand:
        rows.append({
            "Seccion": current_section,
            "Minuto": minuto,
            "Hora": str(h).zfill(2),
            "Dia": dia,
            "Mes": mes,
            "Dia_semana": dia_semana,
            "Comando": comando
        })

df = pd.DataFrame(rows)
df.to_excel("cron_completo.xlsx", index=False)
print("✅ Archivo generado: cron_completo.xlsx")


✅ Archivo generado: cron_completo.xlsx
